In [11]:
import json
import os
import sys
from prettytable import PrettyTable

In [12]:
RESULTS_DIR = "../results/spread_v6_ablation_01/simple_spread_dynamic_hard/pimac_v6_ablation"

ALGORITHMS = [
    "always_off_gate",
    "always_on_gate",
    "no_distill_weight"
]

NAME_MAP = {
    "always_off_gate" : "Always Off Gate",
    "always_on_gate" : "Always On Gate",
    "no_distill_weight" : "No Distillation"
}

SEEDS = [42, 43, 44, 45, 46]

In [13]:
def get_path(algorithm, seed):
    # all files in directory
    files = os.listdir(RESULTS_DIR)
    # find the filename which includes the seed in its name
    files = [f for f in files if algorithm in f]
    files = [f for f in files if f"s{seed}" in f]
    if len(files) == 0:
        raise ValueError(f"No file found for algorithm {algorithm} and seed {seed}")
    elif len(files) > 1:
        raise ValueError(f"Multiple files found for algorithm {algorithm} and seed {seed}: {files}")
    return os.path.join(RESULTS_DIR, files[0])

In [14]:
def make_table(data, task):
    table_top = """
    \\begin{table}[]
    \\centering
    \\caption{{task_name}}
    \\label{{tab:{task_name}}}
    \\begin{tabular}{@{}lccc@{}}
    \\toprule
    Method & Train & Validation & Test \\\\ \\midrule
    """

    table_bottom = """  
    \\bottomrule
    \\end{tabular}
    \\end{table}
    """
    table_top = table_top.replace("{task_name}", task.replace("_", "\\_"))
    rows = []
    for algorithm in data:
        row = f"{NAME_MAP[algorithm]} & {data[algorithm]['train']} & {data[algorithm]['val']} & {data[algorithm]['test']} \\\\"
        rows.append(row)
    return table_top + "\n".join(rows) + table_bottom
    

In [15]:
all_scores = {}
for task in ["spread"]:
    print("--------------------------------------------------")
    print(f"Task: {task}")
    #print("--------------------------------------------------")
    task_scores = {}
    for algorithm in ALGORITHMS:
        #print(f"  Algorithm: {algorithm}")
        scores = {"train": [], "val": [], "test": []}
        for seed in SEEDS:
            path = get_path(algorithm, seed)
            summary_json = os.path.join(path, "summary.json")
            with open(summary_json, "r") as f:
                data = json.load(f)
                train_score = data["test"]["train_counts_mean"]
                val_score = data["test"]["validation_counts_mean"]
                test_score = data["test"]["test_counts_mean"]
                scores["train"].append(train_score)
                scores["val"].append(val_score)
                scores["test"].append(test_score)
        for key in scores:
            mean_score = sum(scores[key]) / len(scores[key])
            std_score = (sum((x - mean_score) ** 2 for x in scores[key]) / (len(scores[key]) - 1)) ** 0.5
            #scores[key] = (mean_score, std_score)
            if task == "lbf_hard":
                mean_score *= 100
                std_score *= 100
            scores[key] = f"${round(mean_score, 2)}$ $\\pm$ ${round(std_score, 2)}$"
        task_scores[algorithm] = scores
    print(make_table(task_scores, task))
    table = PrettyTable()
    table.field_names = ["Method", "Train", "Validation", "Test"]
    for algorithm in ALGORITHMS:
        scores = task_scores[algorithm]
        row = [NAME_MAP[algorithm]]
        for key in ["train", "val", "test"]:
            row.append(scores[key].replace("$", "").replace("\\pm", "±"))
        table.add_row(row)
    #print(table)
    #print("--------------------------------------------------")
    all_scores[task] = task_scores

--------------------------------------------------
Task: spread

    \begin{table}[]
    \centering
    \caption{spread}
    \label{{tab:spread}}
    \begin{tabular}{@{}lccc@{}}
    \toprule
    Method & Train & Validation & Test \\ \midrule
    Always Off Gate & $-41.71$ $\pm$ $0.85$ & $-50.23$ $\pm$ $0.67$ & $-82.95$ $\pm$ $2.34$ \\
Always On Gate & $-41.4$ $\pm$ $0.7$ & $-49.67$ $\pm$ $1.12$ & $-82.2$ $\pm$ $2.13$ \\
No Distillation & $-39.78$ $\pm$ $0.89$ & $-48.24$ $\pm$ $1.09$ & $-80.65$ $\pm$ $2.25$ \\  
    \bottomrule
    \end{tabular}
    \end{table}
    
